## Files API



---

Documentation
https://docs.anthropic.com/en/docs/build-with-claude/files

In [ ]:
!pip install anthropic

In [44]:
from google.colab import files
import anthropic
import os
import datetime
import time
# To view tables in Google Colab
from google.colab import data_table
data_table.enable_dataframe_formatter()

# === API KEY SETUP ===
# Option 1: Retrieve API keys from your saved user data in Colab (safer method)
from google.colab import userdata
api_key = userdata.get('API_KEY')      # Key to access Claude AI


# Option 2: If you don't use userdata, you can directly insert your key
# api_key = "insert-your-API-key-here"

# Instructions for setting up API key in Google Colab:
# 1. Click on the key icon in the left sidebar
# 2. Add a new secret with name 'API_KEY'
# 3. Paste your Anthropic API key as the value

# Option 3: Loading from environment variables
# import os
# api_key = os.environ.get('ANTHROPIC_API_KEY')

# Define which Claude versions to use
CLAUDE_3_7 = "claude-3-7-sonnet-20250219"  # Claude 3.7 version
CLAUDE_4 = "claude-sonnet-4-20250514"     # Claude 4 version
models=[CLAUDE_3_7,CLAUDE_4]

In [87]:
from anthropic import Anthropic, BadRequestError, NotFoundError, PermissionDeniedError
client = Anthropic(
    api_key=api_key
)

In [ ]:
# Upload files using Google UI
uploaded = files.upload()

# Check uploaded file
#print("File uploaded:", list(uploaded.keys()))

In [ ]:
filename = list(uploaded.keys())[0]
print(f"Current file: {filename}")


## 1 - Upload files to your workspace

In [ ]:
#1- upload file
try:
  uploaded_file= client.beta.files.upload(
      file=(filename, open("/content/"+filename, "rb"), "application/pdf"),
  )
  print(f"ID file: {uploaded_file.id}")
  print(f"Nome: {uploaded_file.filename}")
  print(f"Dimensione: {uploaded_file.size_bytes} bytes")
except NotFoundError as e:
    print(f"❌ File non trovato (404): Il file non esiste o non hai accesso")
except anthropic.APIError as e:
    print(f"❌ Errore API: {e}")


## 2 - Using a file in messages API

In [101]:
def prompt_claude(prompt, model_name, file_id):
    """
    Function that asks a Claude model to summarize a document

    Parameters:
    - prompt: the prompt or question to Claude
    - model_name: which version of Claude to use
    -file_id: the id of the file to use

    Returns:
    - duration
    - thinking
    - response: the summary     """
    print(f"Prompt for {model_name}: {prompt} id file {file_id}")

    # Start measuring time
    start_time = time.time()

    # Call Claude with support for files
    response = client.messages.create(
      model=CLAUDE_4,
      max_tokens=1024,
      messages=[
          {
              "role": "user",
              "content": [
                  {
                      "type": "text",
                      "text": prompt #"Please summarize this document for me."
                  },
                  {
                    "type": "document",
                    "source": {
                      "type": "file",
                      "file_id": file_id
                    },

                  }
              ]
          },

    ],
      #REMEMBER TO ADD EXTRA HEADERS SINCE IT IS A BETA FEATURE!!
    extra_headers= {
          "anthropic-beta": "files-api-2025-04-14"
      }
)
    #print(response)
    # Calculate how much time has passed
    end_time = time.time()
    duration = end_time - start_time

    # Extract thinking and response
    thinking = None
    response_text = None

    for block in response.content:
        if block.type == "thinking":
            thinking = block.thinking
            print(f"\nThinking summary: {block.thinking}")
        elif block.type == "text":
            response_text = block.text
            print(f"\nResponse: {block.text}")
    return duration, thinking, response_text

## Using the prompt_claude function

In [ ]:
#client code: we will leverage Claude 4 to read through the doc you uploaded
try:

  duration, thinking, response = prompt_claude("Read carefully this file and provide a concise summary in 5 bullet points.", CLAUDE_4, uploaded_file.id)
  print(duration, thinking, response)
except anthropic.APIError as e:
    print(f"❌ Errore API: {e}")


## 3 - List files and retrieving metadata

In [92]:
my_files = client.beta.files.list()

In [ ]:
#looping from my files and retrieving metadata
my_files
for i in my_files.data:
  print(i.id)

  client.beta.files.retrieve_metadata(i.id)


## 4 - Delete files

In [76]:
#delete a file from your workspace
#result = client.beta.files.delete("file_011CPS86QLMJQEuUgNGNMA9K")

In [96]:
#clean the workspace by deleting all files uploaded
del_files=[]
for i in my_files.data:

  result = client.beta.files.delete(i.id)
  del_files.append(result)

In [ ]:
del_files